In [33]:
# 1. Imports / paths
import json
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown, Code

PROJECT_ROOT = Path.cwd().parents[1]

# Notebook을 project root에서 실행한다고 가정
CANDIDATES_PATH = (
    PROJECT_ROOT
    / "trajectory_analysis"
    / "outputs"
    / "transition_candidates.jsonl"
)

FFF_PATH = (
    PROJECT_ROOT
    / "trajectory_analysis"
    / "outputs"
    / "fff_cases_ranked.jsonl"
)

print("Candidates:", CANDIDATES_PATH)
print("FFF:", FFF_PATH)

assert CANDIDATES_PATH.exists()
assert FFF_PATH.exists()

Candidates: /home/dibaeck/workspace/project_sLM_planning/phase4_method_discovery/trajectory_analysis/outputs/transition_candidates.jsonl
FFF: /home/dibaeck/workspace/project_sLM_planning/phase4_method_discovery/trajectory_analysis/outputs/fff_cases_ranked.jsonl


In [34]:
# 2. Load results
def load_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))

    return rows


candidates = load_jsonl(CANDIDATES_PATH)
fff_cases = load_jsonl(FFF_PATH)

candidate_by_id = {
    row["problem_id"]: row
    for row in candidates
}

fff_by_id = {
    row["problem_id"]: row
    for row in fff_cases
}

print("All candidates:", len(candidates))
print("FFF cases:", len(fff_cases))

All candidates: 100
FFF cases: 89


In [35]:
# 3.Find the unique FPP case
fpp_cases = [
    row
    for row in candidates
    if row["transition"] == "FPP"
]

print("FPP cases:", len(fpp_cases))

for row in fpp_cases:
    print(
        row["problem_id"],
        row["base"]["test_pass_ratio"],
        row["vanilla"]["test_pass_ratio"],
        row["tpr"]["test_pass_ratio"],
    )

FPP cases: 1
deepcoder_taco_00098 0.09090909090909091 1.0 1.0


In [36]:
# 4. Fix qualitative case set
SELECTED_IDS = [
    "deepcoder_taco_00417",  # TPR-only improvement
    "deepcoder_taco_00379",  # Vanilla-only improvement
    "deepcoder_taco_00330",  # TPR-dominant improvement
    "deepcoder_taco_00022",  # TPR degraded
    "deepcoder_taco_00061",  # Vanilla degraded
    "deepcoder_taco_01069",  # Both equal improvement
    "deepcoder_taco_00030",  # Both degraded
    "deepcoder_taco_00098",  # Cell 3에서 확인한 FPP problem_id 추가
]

In [37]:
# 5.Overview table
overview = []

for problem_id in SELECTED_IDS:
    row = candidate_by_id[problem_id]

    overview.append({
        "problem_id": problem_id,
        "transition": row["transition"],

        "Base TPR":
            row["base"]["test_pass_ratio"],

        "Vanilla TPR":
            row["vanilla"]["test_pass_ratio"],

        "TPR TPR":
            row["tpr"]["test_pass_ratio"],

        "Base Status":
            row["base"]["status"],

        "Vanilla Status":
            row["vanilla"]["status"],

        "TPR Status":
            row["tpr"]["status"],
    })

overview_df = pd.DataFrame(overview)
display(overview_df)

,problem_id,transition,Base TPR,Vanilla TPR,TPR TPR,Base Status,Vanilla Status,TPR Status
0,deepcoder_taco_00417,FFF,0.000000,0.000000,0.917808,RUNTIME_ERROR,RUNTIME_ERROR,RUNTIME_ERROR
1,deepcoder_taco_00379,FFF,0.000000,0.686207,0.000000,WRONG_ANSWER,WRONG_ANSWER,WRONG_ANSWER
2,deepcoder_taco_00330,FFF,0.109756,0.256098,0.500000,TIME_LIMIT_EXCEEDED,RUNTIME_ERROR,TIME_LIMIT_EXCEEDED
3,deepcoder_taco_00022,FFF,0.550000,0.550000,0.000000,WRONG_ANSWER,WRONG_ANSWER,WRONG_ANSWER
4,deepcoder_taco_00061,FFF,0.605263,0.157895,0.605263,WRONG_ANSWER,WRONG_ANSWER,WRONG_ANSWER
5,deepcoder_taco_01069,FFF,0.000000,0.512770,0.512770,RUNTIME_ERROR,RUNTIME_ERROR,RUNTIME_ERROR
6,deepcoder_taco_00030,FFF,0.221843,0.023891,0.102389,WRONG_ANSWER,RUNTIME_ERROR,WRONG_ANSWER
7,deepcoder_taco_00098,FPP,0.090909,1.000000,1.000000,WRONG_ANSWER,PASS,PASS


In [38]:
# 6. Helper functions

def show_result_table(row):
    data = []

    for label, key in [
        ("Base", "base"),
        ("Vanilla", "vanilla"),
        ("TPR", "tpr"),
    ]:
        run = row[key]

        data.append({
            "Method": label,
            "Passed": run["passed"],
            "Passed Tests": (
                f"{run['passed_tests']}/{run['total_tests']}"
            ),
            "TPR": run["test_pass_ratio"],
            "Status": run["status"],
        })

    display(pd.DataFrame(data))


def show_plan(label, run):
    display(Markdown(f"### {label} Plan"))

    plan = run.get("plan")

    if plan:
        display(Markdown(plan))
    else:
        print("[Plan not found]")


def show_code(label, run):
    display(Markdown(f"### {label} Code"))

    code = run.get("code")

    if code:
        display(Code(code, language="python"))
    else:
        print("[Code not found]")


def inspect_case(problem_id, show_problem=True, show_codes=True):
    row = candidate_by_id[problem_id]

    display(
        Markdown(
            f"# {problem_id}\n\n"
            f"**Transition:** `{row['transition']}`  \n"
            f"**Category:** "
            f"`{row.get('transition_description', '')}`"
        )
    )

    show_result_table(row)

    if show_problem:
        display(Markdown("## Problem"))
        display(Markdown(f"```\n{row['problem']}\n```"))

    display(Markdown("## Plans"))

    show_plan("Base", row["base"])
    show_plan("Vanilla", row["vanilla"])
    show_plan("TPR", row["tpr"])

    if show_codes:
        display(Markdown("## Generated Codes"))

        show_code("Base", row["base"])
        show_code("Vanilla", row["vanilla"])
        show_code("TPR", row["tpr"])

    return row

### Case01 : 00417

In [39]:
case_00417 = inspect_case(
    "deepcoder_taco_00417"
)

# deepcoder_taco_00417

**Transition:** `FFF`  
**Category:** `persistent_failure`

,Method,Passed,Passed Tests,TPR,Status
0,Base,False,0/73,0.000000,RUNTIME_ERROR
1,Vanilla,False,0/73,0.000000,RUNTIME_ERROR
2,TPR,False,67/73,0.917808,RUNTIME_ERROR


## Problem

```
Sereja loves number sequences very much. That's why he decided to make himself a new one following a certain algorithm.

Sereja takes a blank piece of paper. Then he starts writing out the sequence in m stages. Each time he either adds a new number to the end of the sequence or takes l first elements of the current sequence and adds them c times to the end. More formally, if we represent the current sequence as a_1, a_2, ..., a_{n}, then after we apply the described operation, the sequence transforms into a_1, a_2, ..., a_{n}[, a_1, a_2, ..., a_{l}] (the block in the square brackets must be repeated c times). 

A day has passed and Sereja has completed the sequence. He wonders what are the values of some of its elements. Help Sereja.


-----Input-----

The first line contains integer m (1 ≤ m ≤ 10^5) — the number of stages to build a sequence. 

Next m lines contain the description of the stages in the order they follow. The first number in the line is a type of stage (1 or 2). Type 1 means adding one number to the end of the sequence, in this case the line contains integer x_{i} (1 ≤ x_{i} ≤ 10^5) — the number to add. Type 2 means copying a prefix of length l_{i} to the end c_{i} times, in this case the line further contains two integers l_{i}, c_{i} (1 ≤ l_{i} ≤ 10^5, 1 ≤ c_{i} ≤ 10^4), l_{i} is the length of the prefix, c_{i} is the number of copyings. It is guaranteed that the length of prefix l_{i} is never larger than the current length of the sequence.

The next line contains integer n (1 ≤ n ≤ 10^5) — the number of elements Sereja is interested in. The next line contains the numbers of elements of the final sequence Sereja is interested in. The numbers are given in the strictly increasing order. It is guaranteed that all numbers are strictly larger than zero and do not exceed the length of the resulting sequence. Consider the elements of the final sequence numbered starting from 1 from the beginning to the end of the sequence.

Please, do not use the %lld specifier to read or write 64-bit integers in С++. It is preferred to use the cin, cout streams or the %I64d specifier.


-----Output-----

Print the elements that Sereja is interested in, in the order in which their numbers occur in the input. 


-----Examples-----
Input
6
1 1
1 2
2 2 1
1 3
2 5 2
1 4
16
1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16

Output
1 2 1 2 3 1 2 1 2 3 1 2 1 2 3 4
```

## Plans

...

from collections import deque

def solve():
    m = int(input())
    operations = []
    for _ in range(m):
        op_type = int(input())
        if op_type == 1:
            x = int(input())
            operations.append(('append', x))
        else:
            l, c = map(int, input().split())
            operations.append(('block', l, c))
    
    n = int(input())
    queries = list(map(int, input().split()))
    
    # Initialize the deque with the first element
    sequence = deque([queries[0]])
    
    for op_type, *args in operations:
        if op_type == 'append':
            x = args[0]
            sequence.append(x)
        else:
            l, c = args
            # Remove the last l elements from the deque
            sequence = sequence[-l:]
            # Append the block c times
            sequence.extend(sequence * c)
    
    # Output the results of the queries
    for query in queries:
        print(sequence[query - 1], end=' ')

# Call the solve function
solve()

### Vanilla Code

from collections import deque

def solve():
    m = int(input())
    operations = []
    for _ in range(m):
        op_type = int(input())
        if op_type == 1:
            x = int(input())
            operations.append(('append', x))
        else:
            l, c = map(int, input().split())
            operations.append(('appendleft', (l, c)))
    
    n = int(input())
    queries = list(map(int, input().split()))
    
    # Initialize the deque with the first m elements
    sequence = deque(operations[:m])
    
    # Process each operation
    for op, value in operations[m:]:
        if op == 'append':
            sequence.append(value)
        else:
            l, c = value
            sequence.extendleft(sequence[-l:] * c)
    
    # Query the sequence
    result = [sequence[i - 1] for i in queries]
    print(' '.join(map(str, result)))

solve()

### TPR Code

from collections import deque

def process_operations(m, operations, n, queries):
    sequence = deque()
    
    for op, *args in operations:
        if op == 1:
            x = args[0]
            sequence.append(x)
        elif op == 2:
            l, c = args
            block = [sequence.popleft() for _ in range(l)]
            sequence.extend(block * c)
    
    result = []
    for query in queries:
        result.append(sequence[query - 1])
    
    return result

# Read input
m = int(input())
operations = [list(map(int, input().split())) for _ in range(m)]
n = int(input())
queries = list(map(int, input().split()))

# Process operations and get the result
result = process_operations(m, operations, n, queries)

# Print the result
print(' '.join(map(str, result)))

---

In [40]:
case_00098 = inspect_case(
    "deepcoder_taco_00098"
)


# deepcoder_taco_00098

**Transition:** `FPP`  
**Category:** `common_rl_recovery`

,Method,Passed,Passed Tests,TPR,Status
0,Base,False,12/132,0.090909,WRONG_ANSWER
1,Vanilla,True,132/132,1.000000,PASS
2,TPR,True,132/132,1.000000,PASS


## Problem

```
Takahashi has many red balls and blue balls. Now, he will place them in a row.
Initially, there is no ball placed.
Takahashi, who is very patient, will do the following operation 10^{100} times:
 - Place A blue balls at the end of the row of balls already placed. Then, place B red balls at the end of the row.
How many blue balls will be there among the first N balls in the row of balls made this way?

-----Constraints-----
 - 1 \leq N \leq 10^{18}
 - A, B \geq 0
 - 0 < A + B \leq 10^{18}
 - All values in input are integers.

-----Input-----
Input is given from Standard Input in the following format:
N A B

-----Output-----
Print the number of blue balls that will be there among the first N balls in the row of balls.

-----Sample Input-----
8 3 4

-----Sample Output-----
4

Let b denote a blue ball, and r denote a red ball. The first eight balls in the row will be bbbrrrrb, among which there are four blue balls.
```

## Plans

...

# Read input
N, A, B = map(int, input().split())

# Calculate the number of blue balls
if N <= A:
    print(A)
else:
    print(N % (A + B))

### Vanilla Code

# Read input
N, A, B = map(int, input().split())

# Calculate the number of complete cycles and remaining balls
complete_cycles = N // (A + B)
remaining_balls = N % (A + B)

# Calculate the number of blue balls
blue_balls = complete_cycles * A + min(remaining_balls, A)

# Print the result
print(blue_balls)

### TPR Code

# Read input
N, A, B = map(int, input().split())

# Calculate the number of complete cycles and remaining balls
complete_cycles = N // (A + B)
remaining_balls = N % (A + B)

# Calculate the number of blue balls
blue_balls = complete_cycles * A + min(remaining_balls, A)

# Print the result
print(blue_balls)